## Train Model

In [ ]:


import os, glob, json, random, numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import (
    precision_recall_curve, roc_auc_score, average_precision_score,
    classification_report, confusion_matrix, precision_score, recall_score,
    RocCurveDisplay, PrecisionRecallDisplay
)
from tqdm import tqdm
import matplotlib.pyplot as plt

try:
    from torch.amp import autocast, GradScaler
except Exception:
    from torch.cuda.amp import autocast, GradScaler


# ---------------- Config ----------------
EPOCHS       = 60
BATCH_TRAIN  = 32
BATCH_VAL    = 64
TAU          = 0.50
USE_TTA      = True
TTA_PASSES   = 11
SEEDS        = [42, 77, 123]
PATIENCE     = 12

HEAD_MC      = 6
LBL_SMOOTH   = 0.0001

# Set your folders here
TRAIN_DIR = r"G:\Arafat46\Thesis\Dataset\Advance Model\Data\DS_MaRFW_Art_RealLife\train"
VAL_DIR   = r"G:\Arafat46\Thesis\Dataset\Advance Model\Data\DS_MaRFW_Art_RealLife\test"
#TEST_DIR  = r"G:\Arafat46\Thesis\Dataset\Advance Model\Data\UAP_Dataset\test"  

# ---------------- Utils ----------------
def set_seed(sd=42):
    """Reproducible training."""
    random.seed(sd); np.random.seed(sd)
    torch.manual_seed(sd); torch.cuda.manual_seed_all(sd)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def _npz_has_keys_and_shapes(d):
    """Check .npz structure quickly."""
    return ("pose" in d) and ("rgb" in d) and ("flow" in d) and \
           (d["pose"].ndim==3) and (d["rgb"].ndim==2) and (d["flow"].ndim==2)

def autodetect_dims(root_dir):
    """
    Auto-detect feature dims from one sample.
    (Bangla) Train folder theke ekta file niye F_pose/F_rgb/F_flow detect kore.
    """
    classes = [c for c in sorted(os.listdir(root_dir)) if os.path.isdir(os.path.join(root_dir, c))]
    for cls in classes:
        for fp in glob.glob(os.path.join(root_dir, cls, "*.npz")):
            try:
                d = np.load(fp)
                if not _npz_has_keys_and_shapes(d): continue
                P, R, F = d["pose"], d["rgb"], d["flow"]
                F_pose, F_rgb, F_flow = int(P.shape[2]), int(R.shape[1]), int(F.shape[1])
                print(f"[AutoDetect] sample: {fp}")
                print(f"  pose dim  = {F_pose} (T={P.shape[0]}, K={P.shape[1]})")
                print(f"  rgb  dim  = {F_rgb} (T={R.shape[0]})")
                print(f"  flow dim  = {F_flow} (T={F.shape[0]})")
                return F_pose, F_rgb, F_flow
            except Exception:
                continue
    raise ValueError(f"No readable .npz found in {root_dir}. Check your preprocessing output path.")

def _npz_shapes_ok(d, F_pose, F_rgb, F_flow):
    """Filter out any npz that doesn't match the target dims."""
    try:
        P = d["pose"]; R = d["rgb"]; F = d["flow"]
    except Exception:
        return False
    if P.ndim != 3 or R.ndim != 2 or F.ndim != 2: return False
    return (P.shape[2] == F_pose) and (R.shape[1] == F_rgb) and (F.shape[1] == F_flow)

def compute_stats_npz(root_dir, save_path, F_pose, F_rgb, F_flow):
    """
    Compute per-feature mean/std for normalization with masks (valid frames only).
    """
    ps = np.zeros(F_pose, np.float64); pss=np.zeros(F_pose, np.float64); pn=np.zeros(F_pose, np.int64)
    rs = np.zeros(F_rgb,  np.float64); rss=np.zeros(F_rgb,  np.float64); rn=np.zeros(F_rgb,  np.int64)
    fs = np.zeros(F_flow, np.float64); fss=np.zeros(F_flow, np.float64); fn=np.zeros(F_flow, np.int64)

    classes = [d for d in sorted(os.listdir(root_dir)) if os.path.isdir(os.path.join(root_dir,d))]
    accepted = 0; skipped = 0
    for cls in classes:
        for fp in glob.glob(os.path.join(root_dir, cls, "*.npz")):
            try:
                d = np.load(fp)
            except Exception:
                skipped += 1; continue
            if not _npz_shapes_ok(d, F_pose, F_rgb, F_flow):
                skipped += 1; continue
            accepted += 1

            P = d["pose"].astype(np.float32)
            R = d["rgb"].astype(np.float32)
            Fw = d["flow"].astype(np.float32)
            pmask = (np.abs(P).sum(axis=2) > 0)
            rmask = (np.abs(R).sum(axis=1) > 0)
            fmask = (np.abs(Fw).sum(axis=1) > 0)

            for f in range(F_pose):
                vals = P[:,:,f][pmask]
                if vals.size:
                    ps[f]+=vals.sum(); pss[f]+=(vals*vals).sum(); pn[f]+=vals.size
            if rmask.any():
                vals = R[rmask]
                rs  += vals.sum(axis=0); rss += (vals*vals).sum(axis=0); rn += vals.shape[0]
            if fmask.any():
                vals = Fw[fmask]
                fs  += vals.sum(axis=0); fss += (vals*vals).sum(axis=0); fn += vals.shape[0]

    if accepted == 0:
        raise ValueError(
            f"No files with shapes (pose:*,*,{F_pose}), (rgb:*,{F_rgb}), (flow:*,{F_flow}) found in {root_dir}."
        )

    pmean = ps/np.maximum(pn,1); pvar = pss/np.maximum(pn,1) - pmean*pmean
    rmean = rs/np.maximum(rn,1); rvar = rss/np.maximum(rn,1) - rmean*rmean
    fmean = fs/np.maximum(fn,1); fvar = fss/np.maximum(fn,1) - fmean*fmean
    pstd = np.sqrt(np.clip(pvar, 1e-6, None))
    rstd = np.sqrt(np.clip(rvar, 1e-6, None))
    fstd = np.sqrt(np.clip(fvar, 1e-6, None))

    np.savez(save_path,
             pose_mean=pmean.astype(np.float32), pose_std=pstd.astype(np.float32),
             rgb_mean=rmean.astype(np.float32),  rgb_std=rstd.astype(np.float32),
             flow_mean=fmean.astype(np.float32), flow_std=fstd.astype(np.float32))
    print(f"[Stats] saved {save_path} | accepted={accepted}, skipped={skipped}")
    return save_path

def ensure_stats(root_dir, stats_path, F_pose, F_rgb, F_flow, force=False):
    """Load or recompute stats depending on availability and dim match."""
    need_recompute = force or (not os.path.exists(stats_path))
    if not need_recompute:
        try:
            st = np.load(stats_path)
            ok = (st["pose_mean"].shape[0]==F_pose and st["rgb_mean"].shape[0]==F_rgb and st["flow_mean"].shape[0]==F_flow)
            if ok:
                print(f"[Stats] using existing: {stats_path}"); return stats_path
            else:
                print(f"[Stats] dim mismatch in {stats_path} → recompute"); need_recompute = True
        except Exception as e:
            print(f"[Stats] failed to read {stats_path}: {e} → recompute"); need_recompute = True
    return compute_stats_npz(root_dir, stats_path, F_pose, F_rgb, F_flow)


# ---------------- Dataset ----------------
class MMDataset(Dataset):
    """
    Reads .npz with fixed shapes and applies normalization (using computed stats).
    """
    def __init__(self, root_dir, class_to_idx=None, stats_path=None, train=False,
                 F_pose=117, F_rgb=1280, F_flow=13):
        self.train = train
        self.F_pose, self.F_rgb, self.F_flow = F_pose, F_rgb, F_flow
        self.stats = np.load(stats_path) if stats_path and os.path.exists(stats_path) else None

        if class_to_idx is None:
            classes = [d for d in sorted(os.listdir(root_dir)) if os.path.isdir(os.path.join(root_dir,d))]
            self.class_to_idx = {c:i for i,c in enumerate(classes)}
        else:
            self.class_to_idx = class_to_idx
            classes = list(class_to_idx.keys())

        files, labels = [], []
        skipped = 0
        for cls in classes:
            for f in glob.glob(os.path.join(root_dir, cls, "*.npz")):
                try:
                    d = np.load(f)
                except Exception:
                    skipped += 1; continue
                if not _npz_shapes_ok(d, F_pose, F_rgb, F_flow):
                    skipped += 1; continue
                files.append(f); labels.append(self.class_to_idx[cls])
        if len(files) == 0:
            raise ValueError(f"No usable .npz in {root_dir} after shape filtering. Check preprocessing.")
        self.files, self.labels = files, labels
        print(f"[Dataset:{'train' if train else 'val/test'}] loaded={len(self.files)}, skipped={skipped}")

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        d = np.load(self.files[idx])
        P = d["pose"].astype(np.float32)  # (75,4,117)
        R = d["rgb"].astype(np.float32)   # (75,1280)
        Fw = d["flow"].astype(np.float32) # (75,13)

        if self.stats is not None:
            pm, ps = self.stats["pose_mean"][None,None,:], self.stats["pose_std"][None,None,:]
            rm, rs = self.stats["rgb_mean"][None,:],     self.stats["rgb_std"][None,:]
            fm, fs = self.stats["flow_mean"][None,:],    self.stats["flow_std"][None,:]
            pmask  = (np.abs(P).sum(axis=2) > 0)
            P = (P - pm) / ps
            P *= pmask[:,:,None].astype(np.float32)
            R = (R - rm) / rs
            Fw = (Fw - fm) / fs

        if self.train:
            # light gaussian noise
            P += np.random.normal(0, 0.02, size=P.shape).astype(np.float32)
            R += np.random.normal(0, 0.02, size=R.shape).astype(np.float32)
            Fw+= np.random.normal(0, 0.02, size=Fw.shape).astype(np.float32)

        return (torch.from_numpy(P), torch.from_numpy(R), torch.from_numpy(Fw)), \
               torch.tensor(self.labels[idx], dtype=torch.long)


# ---------------- Model blocks ----------------
def ensure_at_least_one_true(mask_bool):
    """If an entire row is False, set first position True to avoid all-masked attention."""
    if mask_bool is None: return None
    all_false = ~mask_bool.any(dim=1)
    if all_false.any():
        mask_bool = mask_bool.clone()
        mask_bool[all_false, 0] = True
    return mask_bool

class PersonAttention(nn.Module):
    def __init__(self, F, hidden=64):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(F, hidden), nn.ReLU(inplace=True), nn.Linear(hidden, 1))
    def forward(self, x):  # x: (B,T,K,F)
        mask   = (x.abs().sum(dim=-1) > 0)                    # (B,T,K)
        logits = self.mlp(x).squeeze(-1)                      # (B,T,K)
        logits = logits.masked_fill(~mask, -1e4)
        w = torch.softmax(logits, dim=2).unsqueeze(-1)        # (B,T,K,1)
        out = (w * x).sum(dim=2)                              # (B,T,F)
        time_valid = mask.any(dim=2)                          # (B,T)
        time_valid = ensure_at_least_one_true(time_valid)     # safe
        return out, time_valid

class DSConv1DBlock(nn.Module):
    def __init__(self, C, k=5, d=1, expand=1.5, dropout=0.15):
        super().__init__()
        hid = int(C*expand)
        self.dw  = nn.Conv1d(C, C, k, padding=((k-1)//2)*d, dilation=d, groups=C, bias=False)
        self.pw1 = nn.Conv1d(C, hid, 1, bias=False)
        self.pw2 = nn.Conv1d(hid, C, 1, bias=False)
        self.bn  = nn.BatchNorm1d(C)
        self.act = nn.ReLU(inplace=True)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):  # (B,C,T)
        y = self.dw(x)
        y = self.pw2(self.act(self.pw1(y)))
        y = self.bn(y)
        y = self.act(y + x)
        return self.drop(y)

class HybridTemporal(nn.Module):
    """DS-Conv (multi-dilation) + Transformer encoder with safe padding."""
    def __init__(self, dim, conv_layers=2, tx_layers=2, heads=4, dropout=0.15):
        super().__init__()
        self.convs = nn.Sequential(*[
            DSConv1DBlock(dim, k=5, d=2**i, expand=1.5, dropout=dropout)
            for i in range(conv_layers)
        ])
        enc = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, batch_first=True,
            dim_feedforward=dim*4, dropout=dropout, activation='gelu'
        )
        self.tx = nn.TransformerEncoder(enc, num_layers=tx_layers)
        self.ln = nn.LayerNorm(dim)
    def forward(self, x, key_padding_mask=None):   # x: (B,T,dim); key_padding_mask: True=pad
        x = x.transpose(1,2)         # (B,dim,T)
        x = self.convs(x)            # (B,dim,T)
        x = x.transpose(1,2)         # (B,T,dim)
        if key_padding_mask is not None:
            kpm = key_padding_mask
            all_true = kpm.all(dim=1)
            if all_true.any():
                kpm = kpm.clone()
                kpm[all_true, 0] = False
            x = self.tx(x, src_key_padding_mask=kpm)
        else:
            x = self.tx(x)
        return self.ln(x)

class AttnPool(nn.Module):
    def __init__(self, D, heads=4, dropout=0.1):
        super().__init__()
        self.q = nn.Parameter(torch.randn(1,1,D))
        self.attn = nn.MultiheadAttention(D, heads, batch_first=True, dropout=dropout)
    def forward(self, x, time_valid):
        # time_valid: (B,T) True = keep
        tv = ensure_at_least_one_true(time_valid)            # safe
        B = x.size(0); q = self.q.expand(B,-1,-1)
        kpm = ~tv                                           # True = pad
        all_true = kpm.all(dim=1)
        if all_true.any():
            kpm = kpm.clone()
            kpm[all_true, 0] = False
            x = x.clone(); x[all_true, 0, :] = 0.0
        out,_ = self.attn(q, x, x, key_padding_mask=kpm, need_weights=False)
        return out.squeeze(1)

class MMNetV8(nn.Module):
    def __init__(self, F_pose, F_rgb, F_flow, classes=2, tau=0.65, head_mc=HEAD_MC):
        super().__init__()
        self.tau = tau
        self.head_mc = head_mc

        # Pose tower → 256
        self.person = PersonAttention(F_pose, hidden=64)
        self.pose_in = nn.Linear(F_pose, 256)
        self.tem_p   = HybridTemporal(256, conv_layers=2, tx_layers=2, heads=4, dropout=0.15)
        self.gru_p   = nn.GRU(256, 160, batch_first=True, bidirectional=True)      # -> 320
        self.drop_p  = nn.Dropout(0.2)
        self.pool_p  = AttnPool(320, heads=4, dropout=0.1)
        self.proj_p  = nn.Linear(320, 256)
        self.aux_p   = nn.Linear(256, classes)

        # RGB tower (F_rgb → 256)
        self.rgb_in  = nn.Linear(F_rgb, 256)
        self.tem_r   = HybridTemporal(256, conv_layers=2, tx_layers=2, heads=4, dropout=0.15)
        self.gru_r   = nn.GRU(256, 128, batch_first=True, bidirectional=True)      # -> 256
        self.drop_r  = nn.Dropout(0.2)
        self.pool_r  = AttnPool(256, heads=4, dropout=0.1)
        self.aux_r   = nn.Linear(256, classes)

        # Flow tower (13 → 128 → 256)
        self.flow_in = nn.Linear(F_flow, 128)
        self.tem_f   = HybridTemporal(128, conv_layers=2, tx_layers=1, heads=2, dropout=0.15)
        self.gru_f   = nn.GRU(128, 96, batch_first=True, bidirectional=True)       # -> 192
        self.drop_f  = nn.Dropout(0.2)
        self.pool_f  = AttnPool(192, heads=3, dropout=0.1)
        self.proj_f  = nn.Linear(192, 256)
        self.aux_f   = nn.Linear(256, classes)

        # Fusion (gating + pairwise)
        self.gate = nn.Sequential(nn.Linear(256*3, 128), nn.ReLU(inplace=True), nn.Linear(128, 3))
        # Multi-sample dropout head
        self.head_fc1 = nn.Linear(256, 128)
        self.head_do  = nn.ModuleList([nn.Dropout(0.5) for _ in range(self.head_mc)])
        self.head_fc2 = nn.Linear(128, classes)

    def forward_once(self, x_pose, x_rgb, x_flow, return_aux=False):
        # Pose
        xp, tvp = self.person(x_pose)                  # (B,T,Fp), (B,T)
        xp = self.pose_in(xp)
        xp = self.tem_p(xp, key_padding_mask=~tvp)
        xp,_ = self.gru_p(xp); xp = self.drop_p(xp)
        xp = self.pool_p(xp, tvp)
        xp = self.proj_p(xp)
        log_p = self.aux_p(xp)

        # RGB
        tvr = (x_rgb.abs().sum(dim=-1) > 0)
        tvr = ensure_at_least_one_true(tvr)
        xr = self.rgb_in(x_rgb)
        xr = self.tem_r(xr, key_padding_mask=~tvr)
        xr,_ = self.gru_r(xr); xr = self.drop_r(xr)
        xr = self.pool_r(xr, tvr)
        log_r = self.aux_r(xr)

        # Flow
        tvf = (x_flow.abs().sum(dim=-1) > 0)
        tvf = ensure_at_least_one_true(tvf)
        xf = self.flow_in(x_flow)
        xf = self.tem_f(xf, key_padding_mask=~tvf)
        xf,_ = self.gru_f(xf); xf = self.drop_f(xf)
        xf = self.pool_f(xf, tvf)
        xf = self.proj_f(xf)
        log_f = self.aux_f(xf)

        # Fusion (gated + pairwise)
        cat = torch.cat([xp, xr, xf], dim=1)                # (B,768)
        g = torch.softmax(self.gate(cat)/self.tau, dim=1)   # (B,3)
        pair = (xp*xr) + (xp*xf) + (xr*xf)                  # (B,256)
        z = g[:,0:1]*xp + g[:,1:2]*xr + g[:,2:3]*xf + 0.3*pair

        h = F.relu(self.head_fc1(z))
        logits_list = [self.head_fc2(do(h)) for do in self.head_do]
        out = torch.stack(logits_list, 0).mean(0)

        if return_aux:
            return out, (log_p, log_r, log_f)
        return out

    def forward(self, x_pose, x_rgb, x_flow, return_aux=False, rdrop=False):
        if not rdrop:
            return self.forward_once(x_pose, x_rgb, x_flow, return_aux=return_aux)
        # Two stochastic passes for R-Drop
        log1, aux1 = self.forward_once(x_pose, x_rgb, x_flow, return_aux=True)
        log2, aux2 = self.forward_once(x_pose, x_rgb, x_flow, return_aux=True)
        if return_aux:
            return (log1, log2), (aux1, aux2)
        return (log1, log2)

class ComboLoss(nn.Module):
    """CE + focal mixture (stable)."""
    def __init__(self, smooth=LBL_SMOOTH, focal_gamma=2.0, mix=0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=smooth)
        self.gamma = focal_gamma; self.mix = mix
    def forward(self, logits, target):
        ce = self.ce(logits, target)
        with torch.no_grad():
            p = torch.softmax(logits, dim=1)
            pt = p[torch.arange(target.size(0), device=logits.device), target].clamp_(1e-6, 1-1e-6)
        focal = torch.mean(-((1-pt)**self.gamma) * torch.log(pt))
        return (1-self.mix)*ce + self.mix*focal

def js_divergence(p, q):
    m = 0.5*(p+q)
    kl = torch.sum(p*(torch.log(p+1e-8)-torch.log(m+1e-8)), dim=1) + \
         torch.sum(q*(torch.log(q+1e-8)-torch.log(m+1e-8)), dim=1)
    return 0.5*kl.mean()

def rdrop_kl(log1, log2):
    p1 = torch.log_softmax(log1, dim=1); p2 = torch.log_softmax(log2, dim=1)
    q1 = torch.softmax(log1, dim=1);     q2 = torch.softmax(log2, dim=1)
    kl1 = torch.sum(q1*(p1 - p2), dim=1)
    kl2 = torch.sum(q2*(p2 - p1), dim=1)
    return 0.5*(kl1 + kl2).mean()


# ---------------- EMA ----------------
class EMA:
    def __init__(self, model, decay=0.999, device=None):
        self.decay=decay; self.device=device or next(model.parameters()).device
        self.shadow={k:v.detach().clone().to(self.device) for k,v in model.state_dict().items()}
        self.backup=None
    @torch.no_grad()
    def update(self, model):
        for k,v in model.state_dict().items():
            v=v.detach().to(self.device)
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v, alpha=1.0-self.decay)
            else:
                self.shadow[k]=v.clone()
    def apply(self, model):
        self.backup={k:v.detach().clone() for k,v in model.state_dict().items()}
        model.load_state_dict(self.shadow, strict=True)
    def restore(self, model):
        model.load_state_dict(self.backup, strict=True)


# ---------------- Evaluation helpers (with loss) ----------------
@torch.no_grad()
def _eval_from_logits_and_targets(logits_list, y_list):
    """
    Take stacked logits and labels to compute metrics + decision thresholds.
    Returns extra artifacts (cm, probs, y_true) for plotting.
    """
    logits = torch.cat(logits_list, 0)
    y_true = torch.cat(y_list, 0).cpu().numpy()

    probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
    m = np.isfinite(probs)
    if not m.all():
        probs = probs[m]; y_true = y_true[m]

    prec, rec, thr = precision_recall_curve(y_true, probs)
    f1 = (2*prec*rec)/(prec+rec+1e-8); bi = int(np.argmax(f1))
    thr_f1 = float(thr[bi]) if bi < len(thr) else 0.5
    y_pred = (probs>=thr_f1).astype(int)

    best_acc, thr_acc = 0.0, 0.5
    for t in np.linspace(0.0,1.0,1001):
        acc = ((probs>=t).astype(int)==y_true).mean()
        if acc>best_acc: best_acc, thr_acc = acc, float(t)

    p_at = precision_score(y_true, y_pred, zero_division=0)
    r_at = recall_score(y_true, y_pred,   zero_division=0)
    cm   = confusion_matrix(y_true, y_pred)

    return {
        "acc_f1": float((y_pred==y_true).mean()*100.0),
        "acc_best": float(best_acc*100.0),
        "f1": float(f1[bi]) if len(f1) else 0.0,
        "thr_f1": thr_f1, "thr_acc": thr_acc,
        "auroc": float(roc_auc_score(y_true, probs)),
        "auprc": float(average_precision_score(y_true, probs)),
        "report": classification_report(y_true, y_pred, digits=4),
        "cm": cm,
        "precision": float(p_at),
        "recall": float(r_at),
        "probs": probs,
        "y_true": y_true,
    }

@torch.no_grad()
def evaluate_with_loss(loader, model, device, criterion_main):
    """Validation with loss + full metrics (used each epoch)."""
    model.eval()
    logits_list, y_list = [], []
    val_loss = 0.0
    n_batches = 0
    for (xp, xr, xf), y in loader:
        xp, xr, xf = xp.to(device), xr.to(device), xf.to(device)
        y = y.to(device)
        logits = model(xp, xr, xf)
        loss = criterion_main(logits, y)
        val_loss += float(loss.item())
        n_batches += 1
        logits_list.append(logits.float().cpu())
        y_list.append(y.cpu())

    metrics = _eval_from_logits_and_targets(logits_list, y_list)
    metrics["val_loss"] = val_loss / max(1, n_batches)
    return metrics


# ---------------- Training (with history logging + plots) ----------------
def train_once(train_loader, val_loader, device, F_pose, F_rgb, F_flow, tag="s1"):
    set_seed(int(tag[1:]) if tag.startswith('s') else 42)

    model = MMNetV8(F_pose=F_pose, F_rgb=F_rgb, F_flow=F_flow, classes=2, tau=TAU).to(device)
    ema = EMA(model, decay=0.999, device=device)
    criterion_main = ComboLoss(smooth=LBL_SMOOTH, focal_gamma=2.0, mix=0.5)
    criterion_aux  = nn.CrossEntropyLoss(label_smoothing=LBL_SMOOTH)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
    steps = len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=1.0e-3, epochs=EPOCHS, steps_per_epoch=steps,
        pct_start=0.15, anneal_strategy='cos', div_factor=10.0, final_div_factor=10.0
    )
    scaler = GradScaler(enabled=(device.type=='cuda'))

    best_f1, best_acc = -1.0, -1.0
    best_state_f1, best_state_acc = None, None
    no_improve = 0

    # history for plots
    hist = {
        "loss":[], "train_acc":[],
        "val_loss":[], "val_acc_f1":[], "val_acc_best":[],
        "val_f1":[], "val_precision":[], "val_recall":[],
        "val_auroc":[], "val_auprc":[]
    }
    best_epoch_snapshot = None  # for CM/ROC/PR plots

    for ep in range(EPOCHS):
        model.train(); run_loss=0.0; correct=0; total=0
        pbar = tqdm(train_loader, ncols=120, desc=f"[{tag}] Epoch {ep+1}/{EPOCHS}")
        for (xp,xr,xf), y in pbar:
            xp,xr,xf,y = xp.to(device), xr.to(device), xf.to(device), y.to(device)

            # modality dropout (robustness)
            if np.random.rand() < 0.20: xp = torch.zeros_like(xp)  # drop pose
            if np.random.rand() < 0.15: xr = torch.zeros_like(xr)  # drop rgb
            if np.random.rand() < 0.15: xf = torch.zeros_like(xf)  # drop flow

            # mix strategies
            def mixup_batch(xp, xr, xf, y, alpha=0.2):
                if alpha <= 0: return xp, xr, xf, y, None
                lam = np.random.beta(alpha, alpha)
                bsz = y.size(0); idx = torch.randperm(bsz, device=y.device)
                xp_m = lam*xp + (1-lam)*xp[idx]
                xr_m = lam*xr + (1-lam)*xr[idx]
                xf_m = lam*xf + (1-lam)*xf[idx]
                y2 = y[idx]
                return xp_m, xr_m, xf_m, y2, lam

            def time_cutmix_batch(xp, xr, xf, y, p=0.25):
                if np.random.rand()>p: return xp, xr, xf, y, None
                B,T = xp.size(0), xp.size(1)
                idx = torch.randperm(B, device=y.device)
                L = np.random.randint(max(2, T//10), max(3, T//4))
                s = np.random.randint(0, max(1, T-L)); e=s+L
                lam = 1.0 - (L/float(T))
                xp2, xr2, xf2 = xp.clone(), xr.clone(), xf.clone()
                xp2[:, s:e] = xp[idx, s:e]; xr2[:, s:e] = xr[idx, s:e]; xf2[:, s:e] = xf[idx, s:e]
                y2 = y[idx]
                return xp2, xr2, xf2, y2, lam

            mode = np.random.rand()
            if mode < 0.30:
                xp_m, xr_m, xf_m, y2, lam = mixup_batch(xp, xr, xf, y, alpha=0.2)
            elif mode < 0.60:
                xp_m, xr_m, xf_m, y2, lam = time_cutmix_batch(xp, xr, xf, y, p=1.0)
            else:
                xp_m, xr_m, xf_m, y2, lam = xp, xr, xf, None, None

            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type='cuda', enabled=(device.type=='cuda')):
                use_rdrop = (np.random.rand() < 0.5)
                if use_rdrop:
                    (log1, log2), ((lp1,lr1,lf1),(lp2,lr2,lf2)) = model(xp_m, xr_m, xf_m, return_aux=True, rdrop=True)
                    if lam is None:
                        loss_main = 0.5*(criterion_main(log1,y)+criterion_main(log2,y))
                        loss_aux  = 0.1*(criterion_aux(lp1,y)+criterion_aux(lr1,y)+criterion_aux(lf1,y))/3.0 + \
                                    0.1*(criterion_aux(lp2,y)+criterion_aux(lr2,y)+criterion_aux(lf2,y))/3.0
                    else:
                        loss_main = 0.5*(lam*criterion_main(log1,y)+(1-lam)*criterion_main(log1,y2) +
                                         lam*criterion_main(log2,y)+(1-lam)*criterion_main(log2,y2))
                        loss_aux  = 0.1*(lam*(criterion_aux(lp1,y)+criterion_aux(lr1,y)+criterion_aux(lf1,y)) +
                                         (1-lam)*(criterion_aux(lp1,y2)+criterion_aux(lr1,y2)+criterion_aux(lf1,y2)))/3.0 + \
                                    0.1*(lam*(criterion_aux(lp2,y)+criterion_aux(lr2,y)+criterion_aux(lf2,y)) +
                                         (1-lam)*(criterion_aux(lp2,y2)+criterion_aux(lr2,y2)+criterion_aux(lf2,y2)))/3.0
                    loss_cons = rdrop_kl(log1, log2)
                    p_p = torch.softmax(lp1,dim=1); p_r = torch.softmax(lr1,dim=1); p_f = torch.softmax(lf1,dim=1)
                    loss_js = (js_divergence(p_p, p_r)+js_divergence(p_p, p_f)+js_divergence(p_r, p_f))/3.0
                    loss = loss_main + loss_aux + 0.5*loss_cons + 0.05*loss_js
                    logits = log1
                else:
                    logits, (lp,lr,lf) = model(xp_m, xr_m, xf_m, return_aux=True, rdrop=False)
                    if lam is None:
                        loss_main = criterion_main(logits, y)
                        loss_aux  = 0.1*(criterion_aux(lp,y)+criterion_aux(lr,y)+criterion_aux(lf,y))/3.0
                    else:
                        loss_main = lam*criterion_main(logits,y)+(1-lam)*criterion_main(logits,y2)
                        loss_aux  = 0.1*(lam*(criterion_aux(lp,y)+criterion_aux(lr,y)+criterion_aux(lf,y)) +
                                         (1-lam)*(criterion_aux(lp,y2)+criterion_aux(lr,y2)+criterion_aux(lf,y2)))/3.0
                    p_p = torch.softmax(lp,dim=1); p_r = torch.softmax(lr,dim=1); p_f = torch.softmax(lf,dim=1)
                    loss_js = (js_divergence(p_p, p_r)+js_divergence(p_p, p_f)+js_divergence(p_r, p_f))/3.0
                    loss = loss_main + loss_aux + 0.05*loss_js

            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            else:
                loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()

            scheduler.step()
            ema.update(model)

            run_loss += float(loss.item())
            total += y.size(0)
            with torch.no_grad():
                pred = logits.argmax(1); correct += pred.eq(y).sum().item()
            pbar.set_postfix(loss=f"{run_loss/max(1,len(train_loader)):.4f}",
                             acc=f"{100*correct/max(1,total):.2f}%")

        # Train epoch stats
        train_acc_epoch = 100.0 * correct / max(1,total)
        hist["loss"].append(run_loss/max(1,len(train_loader)))
        hist["train_acc"].append(train_acc_epoch)

        # EMA validation with loss + full metrics
        ema.apply(model)
        val = evaluate_with_loss(val_loader, model, device, criterion_main)
        ema.restore(model)

        # log history
        hist["val_loss"].append(val["val_loss"])
        hist["val_acc_f1"].append(val["acc_f1"])
        hist["val_acc_best"].append(val["acc_best"])
        hist["val_f1"].append(val["f1"])
        hist["val_precision"].append(val["precision"])
        hist["val_recall"].append(val["recall"])
        hist["val_auroc"].append(val["auroc"])
        hist["val_auprc"].append(val["auprc"])

        print(f"[{tag}][{ep+1:02d}] "
              f"ValLoss={val['val_loss']:.4f} | ValAcc(F1thr)={val['acc_f1']:.2f}% | "
              f"ValAcc(best)={val['acc_best']:.2f}% | F1={val['f1']:.3f} | "
              f"P={val['precision']:.3f} | R={val['recall']:.3f} | "
              f"AUROC={val['auroc']:.3f} | AUPRC={val['auprc']:.3f}")

        # keep snapshot if improved (for figures)
        improved = False
        if val["f1"] > best_f1 + 1e-4:
            best_f1 = val["f1"]; best_state_f1 = {k: v.clone().cpu() for k,v in ema.shadow.items()}; improved=True
            best_epoch_snapshot = {"cm": val["cm"], "probs": val["probs"], "y_true": val["y_true"]}
        if val["acc_best"] > best_acc + 1e-4:
            best_acc = val["acc_best"]; best_state_acc = {k: v.clone().cpu() for k,v in ema.shadow.items()}; improved=True
            if best_epoch_snapshot is None:
                best_epoch_snapshot = {"cm": val["cm"], "probs": val["probs"], "y_true": val["y_true"]}

        no_improve = 0 if improved else (no_improve+1)
        if no_improve >= PATIENCE:
            print(f"[{tag}] Early stop @ epoch {ep+1}, best F1={best_f1:.3f}, best Acc={best_acc:.2f}%")
            break

    # Load best-F1 EMA state for return
    if best_state_f1 is not None:
        model.load_state_dict(best_state_f1, strict=True)
        print(f"[{tag}] Loaded EMA Best-F1 (F1={best_f1:.3f})")

    # ---------- Save plots ----------
    os.makedirs("runs", exist_ok=True)

    def _plot_and_save(x, y_list, labels, title, fname, ylabel):
        plt.figure()
        for y,lab in zip(y_list, labels):
            plt.plot(x, y, label=lab)
        plt.title(title); plt.xlabel("Epoch"); plt.ylabel(ylabel); plt.legend(); plt.grid(True, alpha=0.3)
        plt.tight_layout(); plt.savefig(os.path.join("runs", fname)); plt.close()

    ep_axis = list(range(1, len(hist["loss"])+1))

    # Train vs Val Loss
    _plot_and_save(ep_axis,
                   [hist["loss"], hist["val_loss"]],
                   ["train loss", "val loss"],
                   f"Loss ({tag})",
                   f"loss_train_val_{tag}.png",
                   "Loss")

    # Accuracy (train + val two variants)
    _plot_and_save(ep_axis, [hist["train_acc"], hist["val_acc_f1"], hist["val_acc_best"]],
                   ["train acc","val acc@F1thr","val acc@best"], f"Accuracy ({tag})", f"acc_{tag}.png", "Accuracy (%)")

    # F1 / AUROC / AUPRC
    _plot_and_save(ep_axis, [hist["val_f1"]], ["val F1"], f"F1 ({tag})", f"f1_{tag}.png", "F1")
    _plot_and_save(ep_axis, [hist["val_auroc"]], ["val AUROC"], f"AUROC ({tag})", f"auroc_{tag}.png", "AUROC")
    _plot_and_save(ep_axis, [hist["val_auprc"]], ["val AUPRC"], f"AUPRC ({tag})", f"auprc_{tag}.png", "AUPRC")

    # Precision / Recall vs epoch
    _plot_and_save(ep_axis,
                   [hist["val_precision"], hist["val_recall"]],
                   ["val precision", "val recall"],
                   f"Precision/Recall ({tag})",
                   f"prec_recall_{tag}.png",
                   "Score")

    # Confusion Matrix heatmap (best snapshot)
    if best_epoch_snapshot is not None:
        cm = best_epoch_snapshot["cm"]
        fig = plt.figure()
        plt.imshow(cm, cmap="Blues"); plt.colorbar()
        plt.xticks([0,1], ["NonFight","Fight"]); plt.yticks([0,1], ["NonFight","Fight"])
        plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"Val Confusion Matrix ({tag})")
        for (i,j),v in np.ndenumerate(cm):
            plt.text(j, i, str(v), ha="center", va="center")
        plt.tight_layout(); plt.savefig(os.path.join("runs", f"cm_{tag}.png")); plt.close(fig)

        # ROC & PR curves
        try:
            y_true = best_epoch_snapshot["y_true"]; probs = best_epoch_snapshot["probs"]
            RocCurveDisplay.from_predictions(y_true, probs)
            plt.title(f"Validation ROC ({tag})"); plt.tight_layout()
            plt.savefig(os.path.join("runs", f"roc_{tag}.png")); plt.close()

            PrecisionRecallDisplay.from_predictions(y_true, probs)
            plt.title(f"Validation PR ({tag})"); plt.tight_layout()
            plt.savefig(os.path.join("runs", f"pr_{tag}.png")); plt.close()
        except Exception:
            pass

    return model, best_state_f1, best_state_acc


# ---------------- loaders ----------------
def build_loaders(TRAIN_DIR, VAL_DIR, stats_path, device, F_pose, F_rgb, F_flow):
    tr_ds = MMDataset(TRAIN_DIR, stats_path=stats_path, train=True,  F_pose=F_pose, F_rgb=F_rgb, F_flow=F_flow)
    va_ds = MMDataset(VAL_DIR,   class_to_idx=tr_ds.class_to_idx, stats_path=stats_path, train=False, F_pose=F_pose, F_rgb=F_rgb, F_flow=F_flow)

    counts = np.bincount(np.array(tr_ds.labels))
    if len(counts)==2 and abs(counts[0]-counts[1]) > 0.1*sum(counts):
        weights = 1.0 / counts
        sample_w = [weights[l] for l in tr_ds.labels]
        sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)
        train_loader = DataLoader(tr_ds, batch_size=BATCH_TRAIN, sampler=sampler, num_workers=0, pin_memory=(device.type=='cuda'))
        print("[Sampler] WeightedRandomSampler enabled.")
    else:
        train_loader = DataLoader(tr_ds, batch_size=BATCH_TRAIN, shuffle=True, num_workers=0, pin_memory=(device.type=='cuda'))

    val_loader = DataLoader(va_ds, batch_size=BATCH_VAL, shuffle=False, num_workers=0, pin_memory=(device.type=='cuda'))
    return train_loader, val_loader, tr_ds.class_to_idx


# ---------------- ensemble eval (optional) ----------------
@torch.no_grad()
def evaluate_ensemble(loader, ckpt_paths, device, F_pose, F_rgb, F_flow):
    models = []
    for cp in ckpt_paths:
        m = MMNetV8(F_pose=F_pose, F_rgb=F_rgb, F_flow=F_flow, classes=2, tau=TAU).to(device)
        state = torch.load(cp, map_location=device)
        m.load_state_dict(state, strict=True)
        m.eval()
        models.append(m)
    probs_all, y_all = [], []
    for (xp,xr,xf), y in loader:
        xp, xr, xf = xp.to(device), xr.to(device), xf.to(device)
        p = []
        for m in models:
            log = m(xp, xr, xf)
            p.append(torch.softmax(log, dim=1)[:,1])
        p = torch.stack(p, 0).mean(0)
        probs_all.append(p.cpu()); y_all.append(y)
    probs = torch.cat(probs_all).numpy(); y_true = torch.cat(y_all).numpy()

    prec, rec, thr = precision_recall_curve(y_true, probs)
    f1 = (2*prec*rec)/(prec+rec+1e-8); bi = int(np.argmax(f1))
    thr_f1 = float(thr[bi]) if bi < len(thr) else 0.5
    y_pred = (probs>=thr_f1).astype(int)

    best_acc, thr_acc = 0.0, 0.5
    for t in np.linspace(0.0,1.0,1001):
        acc = ((probs>=t).astype(int)==y_true).mean()
        if acc>best_acc: best_acc, thr_acc = acc, float(t)

    rep = classification_report(y_true, y_pred, digits=4)
    cm  = confusion_matrix(y_true, y_pred)
    return {
        "acc_f1": float((y_pred==y_true).mean()*100.0),
        "acc_best": float(best_acc*100.0),
        "f1": float(f1[bi]) if len(f1) else 0.0,
        "thr_f1": thr_f1, "thr_acc": thr_acc,
        "auroc": float(roc_auc_score(y_true, probs)),
        "auprc": float(average_precision_score(y_true, probs)),
        "report": rep, "cm": cm
    }


# ---------------- Main ----------------
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    set_seed(42)

    F_POSE, F_RGB, F_FLOW = autodetect_dims(TRAIN_DIR)
    stats_path = ensure_stats(TRAIN_DIR, f"stats_mm_v8_rgb{F_RGB}.npz",
                              F_pose=F_POSE, F_rgb=F_RGB, F_flow=F_FLOW, force=False)

    train_loader, val_loader, class_to_idx = build_loaders(
        TRAIN_DIR, VAL_DIR, stats_path, device, F_POSE, F_RGB, F_FLOW
    )

    best_paths_f1, best_paths_acc = [], []
    for i, sd in enumerate(SEEDS, 1):
        tag = f"s{i}"
        model, st_f1, st_acc = train_once(
            train_loader, val_loader, device, F_POSE, F_RGB, F_FLOW, tag=tag
        )

        if st_f1 is not None:
            fpath = f"mm_v8_best_f1_ema_rgb{F_RGB}_{tag}.pth"
            torch.save(st_f1, fpath); best_paths_f1.append(fpath)
            model.load_state_dict(st_f1, strict=True)
            final_f1 = evaluate_with_loss(val_loader, model, device, ComboLoss())
            print(f"\n[{tag}][Best-F1 EMA]\n{final_f1['report']}")
            print("CM:\n", final_f1["cm"])
            print(f"Acc(F1thr)={final_f1['acc_f1']:.2f}  Acc(best)={final_f1['acc_best']:.2f}  "
                  f"AUROC={final_f1['auroc']:.4f}  AUPRC={final_f1['auprc']:.4f}")

        if st_acc is not None:
            apath = f"mm_v8_best_acc_ema_rgb{F_RGB}_{tag}.pth"
            torch.save(st_acc, apath); best_paths_acc.append(apath)
            model.load_state_dict(st_acc, strict=True)
            final_acc = evaluate_with_loss(val_loader, model, device, ComboLoss())
            print(f"\n[{tag}][Best-Acc EMA]\n{final_acc['report']}")
            print("CM:\n", final_acc["cm"])
            print(f"Acc(F1thr)={final_acc['acc_f1']:.2f}  Acc(best)={final_acc['acc_best']:.2f}  "
                  f"AUROC={final_acc['auroc']:.4f}  AUPRC={final_acc['auprc']:.4f}")

        torch.save(model.state_dict(), f"mm_violence_v8_rgb{F_RGB}_{tag}.pth")

    # TTA on last seed
    if USE_TTA and len(SEEDS) >= 1:
        last_tag = f"s{len(SEEDS)}"
        model = MMNetV8(F_pose=F_POSE, F_rgb=F_RGB, F_flow=F_FLOW, classes=2, tau=TAU).to(device)
        state = torch.load(f"mm_violence_v8_rgb{F_RGB}_{last_tag}.pth", map_location=device)
        model.load_state_dict(state, strict=True)

        # Enable MC Dropout and eval mode together
        model.eval()
        def enable_dropout(m):
            if isinstance(m, nn.Dropout):
                m.train()
        model.apply(enable_dropout)

        # Collect probabilities across multiple stochastic passes
        all_probs = []
        y_true_ref = None
        for _ in range(TTA_PASSES):
            pass_probs = []
            y_collector = []
            for (xp, xr, xf), y in val_loader:
                xp, xr, xf = xp.to(device), xr.to(device), xf.to(device)
                with torch.no_grad():
                    logits = model(xp, xr, xf)
                    prob = torch.softmax(logits, dim=1)[:, 1].detach().cpu()   # <-- detach() fixed
                pass_probs.append(prob)
                y_collector.append(y)

            probs_this_pass = torch.cat(pass_probs, 0).numpy()
            all_probs.append(probs_this_pass)

            # Cache y_true once (same ordering each pass)
            if y_true_ref is None:
                y_true_ref = torch.cat(y_collector, 0).cpu().numpy()

        probs_mean = np.stack(all_probs, 0).mean(0)
        y_true = y_true_ref

        # Metrics @ best-F1 threshold + best-accuracy threshold
        from sklearn.metrics import precision_recall_curve, roc_auc_score, average_precision_score, classification_report, confusion_matrix
        prec, rec, thr = precision_recall_curve(y_true, probs_mean)
        f1 = (2 * prec * rec) / (prec + rec + 1e-8)
        bi = int(np.argmax(f1))
        thr_f1 = float(thr[bi]) if bi < len(thr) else 0.5
        y_pred = (probs_mean >= thr_f1).astype(int)

        best_acc, thr_acc = 0.0, 0.5
        for t in np.linspace(0.0, 1.0, 1001):
            acc = ((probs_mean >= t).astype(int) == y_true).mean()
            if acc > best_acc:
                best_acc, thr_acc = acc, float(t)

        print("\n[TTA Report]\n", classification_report(y_true, y_pred, digits=4))
        print("TTA CM:\n", confusion_matrix(y_true, y_pred))
        print(
            f"TTA Acc(F1thr)={(y_pred==y_true).mean()*100.0:.2f}  "
            f"Acc(best)={best_acc*100.0:.2f}  "
            f"AUROC={roc_auc_score(y_true, probs_mean):.4f}  "
            f"AUPRC={average_precision_score(y_true, probs_mean):.4f}"
        )


    # Ensemble (Best-F1 paths)
    if len(best_paths_f1) >= 2:
        ens = evaluate_ensemble(val_loader, best_paths_f1, device, F_POSE, F_RGB, F_FLOW)
        print("\n[Ensemble (Best-F1 EMA) Report]\n", ens["report"])
        print("Ensemble CM:\n", ens["cm"])
        print(f"Ensemble Acc(F1thr)={ens['acc_f1']:.2f}  Acc(best)={ens['acc_best']:.2f}  "
              f"AUROC={ens['auroc']:.4f}  AUPRC={ens['auprc']:.4f}")

    with open("label_mapping.json", "w") as f: json.dump(class_to_idx, f, indent=2)
    print("\n✅ Saved checkpoints & mapping.")
    print("Files:")
    print(f" - stats_mm_v8_rgb{F_RGB}.npz, label_mapping.json")
    print(f" - mm_v8_best_f1_ema_rgb{F_RGB}_s*.pth (EMA Best-F1)")
    print(f" - mm_v8_best_acc_ema_rgb{F_RGB}_s*.pth (EMA Best-Acc)")
    print(f" - mm_violence_v8_rgb{F_RGB}_s*.pth (deploy)")
    print(" - runs/*.png (training curves & images)")

if __name__ == "__main__":
    main()


[AutoDetect] sample: G:\Arafat46\Thesis\Dataset\Advance Model\Data\DS_MaRFW_Art_RealLife\train\Fight\fight_0001.npz
  pose dim  = 117 (T=75, K=4)
  rgb  dim  = 1280 (T=75)
  flow dim  = 13 (T=75)
[Stats] saved stats_mm_v8_rgb1280.npz | accepted=9605, skipped=0
[Dataset:train] loaded=9605, skipped=0
[Dataset:val/test] loaded=1861, skipped=0


[s1] Epoch 1/60: 100%|███████████████████████████████████████| 301/301 [01:04<00:00,  4.69it/s, acc=78.68%, loss=0.3907]
g:\conda_envs\torch-arafat-vio\lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(


[s1][01] ValLoss=0.4181 | ValAcc(F1thr)=77.11% | ValAcc(best)=77.54% | F1=0.779 | P=0.742 | R=0.820 | AUROC=0.841 | AUPRC=0.846


[s1] Epoch 2/60: 100%|███████████████████████████████████████| 301/301 [00:54<00:00,  5.49it/s, acc=83.24%, loss=0.3240]


[s1][02] ValLoss=0.3624 | ValAcc(F1thr)=84.36% | ValAcc(best)=85.06% | F1=0.842 | P=0.836 | R=0.848 | AUROC=0.913 | AUPRC=0.921


...

[s3][32] ValLoss=0.1458 | ValAcc(F1thr)=92.53% | ValAcc(best)=92.53% | F1=0.924 | P=0.925 | R=0.923 | AUROC=0.976 | AUPRC=0.978
[s3] Early stop @ epoch 32, best F1=0.940, best Acc=94.14%
[s3] Loaded EMA Best-F1 (F1=0.940)

[s3][Best-F1 EMA]
              precision    recall  f1-score   support

           0     0.9382    0.9471    0.9427       946
           1     0.9448    0.9355    0.9401       915

    accuracy                         0.9414      1861
   macro avg     0.9415    0.9413    0.9414      1861
weighted avg     0.9415    0.9414    0.9414      1861

CM:
 [[896  50]
 [ 59 856]]
Acc(F1thr)=94.14  Acc(best)=94.14  AUROC=0.9829  AUPRC=0.9843

[s3][Best-Acc EMA]
              precision    recall  f1-score   support

           0     0.9382    0.9471    0.9427       946
           1     0.9448    0.9355    0.9401       915

    accuracy                         0.9414      1861
   macro avg     0.9415    0.9413    0.9414      1861
weighted avg     0.9415    0.9414    0.9414      1

C:\Users\USER\AppData\Local\Temp\ipykernel_36532\4029350919.py:856: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(f"mm_violence_v8_rgb{F_RGB}_{last_tag}.p


[TTA Report]
               precision    recall  f1-score   support

           0     0.9411    0.9461    0.9436       946
           1     0.9440    0.9388    0.9414       915

    accuracy                         0.9425      1861
   macro avg     0.9425    0.9424    0.9425      1861
weighted avg     0.9425    0.9425    0.9425      1861

TTA CM:
 [[895  51]
 [ 56 859]]
TTA Acc(F1thr)=94.25  Acc(best)=94.25  AUROC=0.9828  AUPRC=0.9843


C:\Users\USER\AppData\Local\Temp\ipykernel_36532\4029350919.py:772: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(cp, map_location=device)



[Ensemble (Best-F1 EMA) Report]
               precision    recall  f1-score   support

           0     0.9325    0.9641    0.9480       946
           1     0.9615    0.9279    0.9444       915

    accuracy                         0.9463      1861
   macro avg     0.9470    0.9460    0.9462      1861
weighted avg     0.9468    0.9463    0.9462      1861

Ensemble CM:
 [[912  34]
 [ 66 849]]
Ensemble Acc(F1thr)=94.63  Acc(best)=94.63  AUROC=0.9860  AUPRC=0.9875

✅ Saved checkpoints & mapping.
Files:
 - stats_mm_v8_rgb1280.npz, label_mapping.json
 - mm_v8_best_f1_ema_rgb1280_s*.pth (EMA Best-F1)
 - mm_v8_best_acc_ema_rgb1280_s*.pth (EMA Best-Acc)
 - mm_violence_v8_rgb1280_s*.pth (deploy)
 - runs/*.png (training curves & images)


## Test

In [2]:
# -*- coding: utf-8 -*-


import os, glob, json, numpy as np
from datetime import datetime
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    precision_recall_curve, roc_auc_score, average_precision_score,
    classification_report, confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay
)
import matplotlib.pyplot as plt

# =======================
# ======= CONFIGS =======


DEFAULT_TEST_DIR = r"G:\Arafat46\Thesis\Dataset\Advance Model\Data\DS_MaRFW_Art_RealLife\test" 

DEFAULT_CKPT     = ""   # keep empty -> auto-pick latest best-F1/Acc or deploy
DEFAULT_STATS    = ""   # keep empty -> auto-pick stats_mm_v8_rgb*.npz
DEFAULT_LABELS   = "label_mapping.json"
DEFAULT_BATCH    = 64
DEFAULT_TTA      = False
DEFAULT_TTA_PASSES = 11
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# ===== Model blocks ======
def ensure_at_least_one_true(mask_bool):
    if mask_bool is None: return None
    all_false = ~mask_bool.any(dim=1)
    if all_false.any():
        mask_bool = mask_bool.clone()
        mask_bool[all_false, 0] = True
    return mask_bool

class PersonAttention(nn.Module):
    def __init__(self, F, hidden=64):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(F, hidden), nn.ReLU(inplace=True), nn.Linear(hidden, 1))
    def forward(self, x):  # x: (B,T,K,F)
        mask   = (x.abs().sum(dim=-1) > 0)  # (B,T,K)
        logits = self.mlp(x).squeeze(-1)    # (B,T,K)
        logits = logits.masked_fill(~mask, -1e4)
        w = torch.softmax(logits, dim=2).unsqueeze(-1)  # (B,T,K,1)
        out = (w * x).sum(dim=2)                        # (B,T,F)
        time_valid = ensure_at_least_one_true(mask.any(dim=2))  # (B,T)
        return out, time_valid

class DSConv1DBlock(nn.Module):
    def __init__(self, C, k=5, d=1, expand=1.5, dropout=0.15):
        super().__init__()
        hid = int(C*expand)
        self.dw  = nn.Conv1d(C, C, k, padding=((k-1)//2)*d, dilation=d, groups=C, bias=False)
        self.pw1 = nn.Conv1d(C, hid, 1, bias=False)
        self.pw2 = nn.Conv1d(hid, C, 1, bias=False)
        self.bn  = nn.BatchNorm1d(C)
        self.act = nn.ReLU(inplace=True); self.drop = nn.Dropout(dropout)
    def forward(self, x):  # (B,C,T)
        y = self.dw(x)
        y = self.pw2(self.act(self.pw1(y)))
        y = self.bn(y)
        y = self.act(y + x)
        return self.drop(y)

class HybridTemporal(nn.Module):
    def __init__(self, dim, conv_layers=2, tx_layers=2, heads=4, dropout=0.15):
        super().__init__()
        self.convs = nn.Sequential(*[DSConv1DBlock(dim, k=5, d=2**i, expand=1.5, dropout=dropout) for i in range(conv_layers)])
        enc = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, batch_first=True,
                                         dim_feedforward=dim*4, dropout=dropout, activation='gelu')
        self.tx = nn.TransformerEncoder(enc, num_layers=tx_layers)
        self.ln = nn.LayerNorm(dim)
    def forward(self, x, key_padding_mask=None):   # x: (B,T,dim); key_padding_mask: True=pad
        x = x.transpose(1,2)         # (B,dim,T)
        x = self.convs(x)            # (B,dim,T)
        x = x.transpose(1,2)         # (B,T,dim)
        if key_padding_mask is not None:
            kpm = key_padding_mask
            all_true = kpm.all(dim=1)
            if all_true.any():
                kpm = kpm.clone()
                kpm[all_true, 0] = False
            x = self.tx(x, src_key_padding_mask=kpm)
        else:
            x = self.tx(x)
        return self.ln(x)

class AttnPool(nn.Module):
    def __init__(self, D, heads=4, dropout=0.1):
        super().__init__()
        self.q = nn.Parameter(torch.randn(1,1,D))
        self.attn = nn.MultiheadAttention(D, heads, batch_first=True, dropout=dropout)
    def forward(self, x, time_valid):
        tv = ensure_at_least_one_true(time_valid)            # (B,T)
        B = x.size(0); q = self.q.expand(B,-1,-1)
        kpm = ~tv
        all_true = kpm.all(dim=1)
        if all_true.any():
            kpm = kpm.clone()
            kpm[all_true, 0] = False
            x = x.clone()
            x[all_true, 0, :] = 0.0
        out,_ = self.attn(q, x, x, key_padding_mask=kpm, need_weights=False)
        return out.squeeze(1)

class MMNetV8(nn.Module):
    def __init__(self, F_pose, F_rgb, F_flow, classes=2, tau=0.65, head_mc=6):
        super().__init__()
        self.tau = tau
        self.head_mc = head_mc

        # Pose tower -> 256
        self.person = PersonAttention(F_pose, hidden=64)
        self.pose_in = nn.Linear(F_pose, 256)
        self.tem_p   = HybridTemporal(256, conv_layers=2, tx_layers=2, heads=4, dropout=0.15)
        self.gru_p   = nn.GRU(256, 160, batch_first=True, bidirectional=True)      # -> 320
        self.drop_p  = nn.Dropout(0.2)
        self.pool_p  = AttnPool(320, heads=4, dropout=0.1)
        self.proj_p  = nn.Linear(320, 256)
        self.aux_p   = nn.Linear(256, classes)

        # RGB tower -> 256
        self.rgb_in  = nn.Linear(F_rgb, 256)
        self.tem_r   = HybridTemporal(256, conv_layers=2, tx_layers=2, heads=4, dropout=0.15)
        self.gru_r   = nn.GRU(256, 128, batch_first=True, bidirectional=True)      # -> 256
        self.drop_r  = nn.Dropout(0.2)
        self.pool_r  = AttnPool(256, heads=4, dropout=0.1)
        self.aux_r   = nn.Linear(256, classes)

        # Flow tower -> 128 -> 256
        self.flow_in = nn.Linear(F_flow, 128)
        self.tem_f   = HybridTemporal(128, conv_layers=2, tx_layers=1, heads=2, dropout=0.15)
        self.gru_f   = nn.GRU(128, 96, batch_first=True, bidirectional=True)       # -> 192
        self.drop_f  = nn.Dropout(0.2)
        self.pool_f  = AttnPool(192, heads=3, dropout=0.1)
        self.proj_f  = nn.Linear(192, 256)
        self.aux_f   = nn.Linear(256, classes)

        # Fusion
        self.gate = nn.Sequential(nn.Linear(256*3, 128), nn.ReLU(inplace=True), nn.Linear(128, 3))
        self.head_fc1 = nn.Linear(256, 128)
        self.head_do  = nn.ModuleList([nn.Dropout(0.5) for _ in range(self.head_mc)])
        self.head_fc2 = nn.Linear(128, classes)

    def forward_once(self, x_pose, x_rgb, x_flow):
        # Pose
        xp, tvp = self.person(x_pose)                  # (B,T,Fp), (B,T)
        xp = self.pose_in(xp)
        xp = self.tem_p(xp, key_padding_mask=~tvp)
        xp,_ = self.gru_p(xp); xp = self.drop_p(xp)
        xp = self.pool_p(xp, tvp)
        xp = self.proj_p(xp)
        log_p = self.aux_p(xp)

        # RGB
        tvr = ensure_at_least_one_true(x_rgb.abs().sum(dim=-1) > 0)
        xr = self.rgb_in(x_rgb)
        xr = self.tem_r(xr, key_padding_mask=~tvr)
        xr,_ = self.gru_r(xr); xr = self.drop_r(xr)
        xr = self.pool_r(xr, tvr)
        log_r = self.aux_r(xr)

        # Flow
        tvf = ensure_at_least_one_true(x_flow.abs().sum(dim=-1) > 0)
        xf = self.flow_in(x_flow)
        xf = self.tem_f(xf, key_padding_mask=~tvf)
        xf,_ = self.gru_f(xf); xf = self.drop_f(xf)
        xf = self.pool_f(xf, tvf)
        xf = self.proj_f(xf)
        log_f = self.aux_f(xf)

        # Fusion
        cat = torch.cat([xp, xr, xf], dim=1)
        g = torch.softmax(self.gate(cat)/self.tau, dim=1)
        pair = (xp*xr) + (xp*xf) + (xr*xf)
        z = g[:,0:1]*xp + g[:,1:2]*xr + g[:,2:3]*xf + 0.3*pair

        h = F.relu(self.head_fc1(z))
        logits_list = [self.head_fc2(do(h)) for do in self.head_do]
        out = torch.stack(logits_list, 0).mean(0)
        return out

    def forward(self, x_pose, x_rgb, x_flow):
        return self.forward_once(x_pose, x_rgb, x_flow)

# ============= Data =============
def _npz_shapes_ok(d, F_pose, F_rgb, F_flow):
    return ("pose" in d and "rgb" in d and "flow" in d and
            d["pose"].ndim==3 and d["rgb"].ndim==2 and d["flow"].ndim==2 and
            d["pose"].shape[2]==F_pose and d["rgb"].shape[1]==F_rgb and d["flow"].shape[1]==F_flow)

def find_latest(patterns):
    cand = []
    for pat in patterns: cand += glob.glob(pat)
    if not cand: return None
    cand.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return cand[0]

class TestDataset(Dataset):
    def __init__(self, root_dir, stats_path, class_to_idx=None, F_pose=117, F_rgb=1280, F_flow=13):
        if class_to_idx is None:
            classes = [d for d in sorted(os.listdir(root_dir)) if os.path.isdir(os.path.join(root_dir,d))]
            self.class_to_idx = {c:i for i,c in enumerate(classes)}
        else:
            self.class_to_idx = class_to_idx
        self.stats = np.load(stats_path)
        self.F_pose, self.F_rgb, self.F_flow = F_pose, F_rgb, F_flow

        files, labels = [], []
        for cls, idx in self.class_to_idx.items():
            for f in glob.glob(os.path.join(root_dir, cls, "*.npz")):
                try:
                    d = np.load(f)
                    if _npz_shapes_ok(d, F_pose, F_rgb, F_flow):
                        files.append(f); labels.append(idx)
                except Exception:
                    continue
        if not files:
            raise ValueError("No valid npz found in test directory for given dims/stats.")
        self.files, self.labels = files, labels
        print(f"[TestDataset] loaded={len(self.files)}")

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        d = np.load(self.files[idx])
        P  = d["pose"].astype(np.float32)
        R  = d["rgb"].astype(np.float32)
        Fw = d["flow"].astype(np.float32)

        pm, ps = self.stats["pose_mean"][None,None,:], self.stats["pose_std"][None,None,:]
        rm, rs = self.stats["rgb_mean"][None,:],     self.stats["rgb_std"][None,:]
        fm, fs = self.stats["flow_mean"][None,:],    self.stats["flow_std"][None,:]

        pmask  = (np.abs(P).sum(axis=2) > 0)
        P = (P - pm) / ps
        P *= pmask[:,:,None].astype(np.float32)
        R = (R - rm) / rs
        Fw = (Fw - fm) / fs

        y = self.labels[idx]
        return (torch.from_numpy(P), torch.from_numpy(R), torch.from_numpy(Fw)), torch.tensor(y, dtype=torch.long)

# ========== Eval helpers ==========
@torch.no_grad()
def _eval_from_probs(probs, y_true):
    m = np.isfinite(probs)
    if not m.all():
        probs = probs[m]; y_true = y_true[m]
    prec, rec, thr = precision_recall_curve(y_true, probs)
    f1 = (2*prec*rec)/(prec+rec+1e-8); bi = int(np.argmax(f1))
    thr_f1 = float(thr[bi]) if bi < len(thr) else 0.5
    y_pred = (probs>=thr_f1).astype(int)
    best_acc, thr_acc = 0.0, 0.5
    for t in np.linspace(0.0,1.0,1001):
        acc = ((probs>=t).astype(int)==y_true).mean()
        if acc>best_acc: best_acc, thr_acc = acc, float(t)
    return {
        "acc_f1": float((y_pred==y_true).mean()*100.0),
        "acc_best": float(best_acc*100.0),
        "f1": float(f1[bi]) if len(f1) else 0.0,
        "thr_f1": thr_f1, "thr_acc": thr_acc,
        "auroc": float(roc_auc_score(y_true, probs)),
        "auprc": float(average_precision_score(y_true, probs)),
        "report": classification_report(y_true, y_pred, digits=4),
        "cm": confusion_matrix(y_true, y_pred),
        "probs": probs, "y_true": y_true
    }

@torch.no_grad()
def evaluate(loader, model, device):
    model.eval()
    probs_list, y_list = [], []
    for (xp, xr, xf), y in loader:
        xp, xr, xf = xp.to(device), xr.to(device), xf.to(device)
        logits = model(xp, xr, xf)
        probs_list.append(torch.softmax(logits, dim=1)[:,1].cpu())
        y_list.append(y)
    probs = torch.cat(probs_list).numpy()
    y_true = torch.cat(y_list).numpy()
    return _eval_from_probs(probs, y_true)

@torch.no_grad()
def evaluate_tta(loader, model, device, passes=11):
    model.eval()
    def enable_dropout(m):
        if isinstance(m, nn.Dropout): m.train()
    model.apply(enable_dropout)

    probs_all, y_all = [], None
    for _ in range(passes):
        probs_list, y_list = [], []
        for (xp, xr, xf), y in loader:
            xp, xr, xf = xp.to(device), xr.to(device), xf.to(device)
            logits = model(xp, xr, xf)
            probs_list.append(torch.softmax(logits, dim=1)[:,1].cpu())
            y_list.append(y)
        probs_all.append(torch.cat(probs_list)); y_all = torch.cat(y_list)

    probs = torch.stack(probs_all, 0).mean(0).numpy()
    y_true = y_all.numpy()
    return _eval_from_probs(probs, y_true)

def save_figures(eval_res, out_dir, title_prefix="TEST"):
    os.makedirs(out_dir, exist_ok=True)
    y_true = eval_res["y_true"]; probs = eval_res["probs"]
    # ROC
    try:
        RocCurveDisplay.from_predictions(y_true, probs)
        plt.title(f"{title_prefix} ROC (AUROC={eval_res['auroc']:.4f})")
        plt.savefig(os.path.join(out_dir, f"{title_prefix.lower()}_roc.png"), bbox_inches="tight")
        plt.close()
    except Exception:
        pass
    # PR
    try:
        PrecisionRecallDisplay.from_predictions(y_true, probs)
        plt.title(f"{title_prefix} PR (AUPRC={eval_res['auprc']:.4f})")
        plt.savefig(os.path.join(out_dir, f"{title_prefix.lower()}_pr.png"), bbox_inches="tight")
        plt.close()
    except Exception:
        pass
    # Confusion Matrix
    try:
        cm = eval_res["cm"]
        fig = plt.figure()
        plt.imshow(cm, cmap="Blues"); plt.colorbar()
        plt.xticks([0,1], ["NonFight","Fight"]); plt.yticks([0,1], ["NonFight","Fight"])
        plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"{title_prefix} Confusion Matrix")
        for (i,j),v in np.ndenumerate(cm):
            plt.text(j, i, str(v), ha="center", va="center")
        plt.savefig(os.path.join(out_dir, f"{title_prefix.lower()}_cm.png"), bbox_inches="tight")
        plt.close(fig)
    except Exception:
        pass

# ========== Runner (no argparse) ==========
def run_test(
    TEST_DIR=DEFAULT_TEST_DIR,
    CKPT=DEFAULT_CKPT,
    STATS=DEFAULT_STATS,
    LABELS=DEFAULT_LABELS,
    BATCH=DEFAULT_BATCH,
    TTA=DEFAULT_TTA,
    TTA_PASSES=DEFAULT_TTA_PASSES,
    DEVICE_STR=DEVICE,
):
    device = torch.device(DEVICE_STR)

    # Autopick stats & checkpoint if not provided
    if not STATS:
        STATS = find_latest(["stats_mm_v8_rgb*.npz"])
        if not STATS: raise FileNotFoundError("No stats_mm_v8_rgb*.npz found; set DEFAULT_STATS path.")
    if not CKPT:
        CKPT = find_latest([
            "mm_v8_best_f1_ema_rgb*_s*.pth",
            "mm_v8_best_acc_ema_rgb*_s*.pth",
            "mm_violence_v8_rgb*_s*.pth"
        ])
        if not CKPT: raise FileNotFoundError("No checkpoint found; set DEFAULT_CKPT path.")

    st = np.load(STATS)
    F_pose = int(st["pose_mean"].shape[0]); F_rgb = int(st["rgb_mean"].shape[0]); F_flow = int(st["flow_mean"].shape[0])
    print(f"[Dims] pose={F_pose}, rgb={F_rgb}, flow={F_flow}")
    print(f"[Paths] stats={STATS}")
    print(f"[Paths] ckpt ={CKPT}")
    print(f"[Paths] test ={TEST_DIR}")

    # Label mapping
    class_to_idx = None
    if os.path.exists(LABELS):
        try:
            with open(LABELS, "r") as f: class_to_idx = json.load(f)
            print(f"[LabelMap] {LABELS}: {class_to_idx}")
        except Exception:
            print("[LabelMap] failed to read; inferring from test subfolders.")

    # Dataset & loader
    te_ds = TestDataset(TEST_DIR, STATS, class_to_idx=class_to_idx, F_pose=F_pose, F_rgb=F_rgb, F_flow=F_flow)
    te_loader = DataLoader(te_ds, batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=(device.type=='cuda'))

    # Model
    model = MMNetV8(F_pose=F_pose, F_rgb=F_rgb, F_flow=F_flow, classes=2, tau=0.65, head_mc=6).to(device)
    state = torch.load(CKPT, map_location=device)
    if isinstance(state, dict) and all(isinstance(v, torch.Tensor) for v in state.values()):
        model.load_state_dict(state, strict=True)
    else:
        sd = state.get("state_dict", state.get("model", None))
        if sd is None: raise RuntimeError("Checkpoint format not recognized.")
        model.load_state_dict(sd, strict=True)

    # Eval
    res = evaluate_tta(te_loader, model, device, passes=TTA_PASSES) if TTA else evaluate(te_loader, model, device)
    tag = "TEST_TTA" if TTA else "TEST"

    print(f"\n[{tag} Report]\n{res['report']}")
    print("Confusion Matrix:\n", res["cm"])
    print(f"Acc(F1thr)={res['acc_f1']:.2f}%  Acc(best)={res['acc_best']:.2f}%  "
          f"AUROC={res['auroc']:.4f}  AUPRC={res['auprc']:.4f}")
    print(f"Best thresholds -> F1_thr={res['thr_f1']:.4f}  Acc_thr={res['thr_acc']:.4f}")

    out_dir = os.path.join("runs_test", datetime.now().strftime("%Y%m%d_%H%M%S"))
    save_figures(res, out_dir, title_prefix=tag)
    print(f"\nSaved figures to: {out_dir}")
    print(f"- {tag.lower()}_roc.png, {tag.lower()}_pr.png, {tag.lower()}_cm.png")

if __name__ == "__main__":
   
    # Change constants at the top if needed; or programmatically call run_test(...).
    run_test(
        TEST_DIR=DEFAULT_TEST_DIR,
        CKPT=DEFAULT_CKPT,        # "" => auto-pick latest best
        STATS=DEFAULT_STATS,      # "" => auto-pick latest stats
        LABELS=DEFAULT_LABELS,
        BATCH=DEFAULT_BATCH,
        TTA=DEFAULT_TTA,
        TTA_PASSES=DEFAULT_TTA_PASSES,
        DEVICE_STR=DEVICE,
    )


[Dims] pose=117, rgb=1280, flow=13
[Paths] stats=stats_mm_v8_rgb1280.npz
[Paths] ckpt =mm_violence_v8_rgb1280_s3.pth
[Paths] test =G:\Arafat46\Thesis\Dataset\Advance Model\Data\DS_MaRFW_Art_RealLife\test
[LabelMap] label_mapping.json: {'Fight': 0, 'NonFight': 1}
[TestDataset] loaded=1861


C:\Users\USER\AppData\Local\Temp\ipykernel_36532\3595346569.py:387: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(CKPT, map_location=device)



[TEST Report]
              precision    recall  f1-score   support

           0     0.9391    0.9461    0.9426       946
           1     0.9438    0.9366    0.9402       915

    accuracy                         0.9414      1861
   macro avg     0.9415    0.9414    0.9414      1861
weighted avg     0.9414    0.9414    0.9414      1861

Confusion Matrix:
 [[895  51]
 [ 58 857]]
Acc(F1thr)=94.14%  Acc(best)=94.14%  AUROC=0.9829  AUPRC=0.9843
Best thresholds -> F1_thr=0.3857  Acc_thr=0.3800

Saved figures to: runs_test\20251116_201331
- test_roc.png, test_pr.png, test_cm.png
